# Análise de dados TCP-CI

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,LSEKEENMVKSQ,338,349,12,HLA-A*23:01,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36986,1,LSEKEENMVKSQ,338,349,12,HLA-A*24:02,1367,100.00,LEENMVKSQ,LSEKEENMVKSQ,0.000000,100.00
36987,1,LSEKEENMVKSQ,338,349,12,HLA-A*32:01,1367,100.00,LSEKEVKSQ,LSEKEENMVKSQ,0.000000,100.00
36988,1,EKEENMVKSQVT,340,351,12,HLA-A*11:01,1369,100.00,ENMVKSQVT,EKEENMVKSQVT,0.000000,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [3]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2606,1,CLWPKTHTLW,223,232,10,HLA-B*44:03,567,4.90,CLWPKTHLW,CLWPKTHTLW,0.003115,4.90
2607,1,NELNYVLWE,73,81,9,HLA-B*44:03,73,4.90,NELNYVLWE,NELNYVLWE,0.003085,4.90
2608,1,DQKAVHADMGY,190,200,11,HLA-B*44:02,877,4.90,DQKAVHMGY,DQKAVHADMGY,0.002847,4.90
2609,1,TPPVSDLKY,105,113,9,HLA-B*44:02,105,4.90,TPPVSDLKY,TPPVSDLKY,0.002830,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAV,186,194,7,2.700,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*6..."
1,AAIKDQKAVH,186,195,2,4.200,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESSK,196,206,2,3.100,"HLA-A*03:01, HLA-A*11:01"
3,AGDVKGVLTK,90,99,3,1.400,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01"
4,AGPFSQHNY,248,256,8,2.300,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
...,...,...,...,...,...,...
593,YVLWEGGHDL,77,86,8,3.150,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*3..."
594,YVLWEGGHDLT,77,87,2,2.050,"HLA-A*02:01, HLA-A*02:06"
595,YVLWEGGHDLTV,77,88,4,2.750,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-B*4..."
596,YWIESSKNQTW,200,210,12,0.775,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."


# Filtragem por epítopos presentes em mais de determinada quantidade de alelos.

In [5]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 10
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ALTPPVSDL,103,111,11,2.200,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
1,ALTPPVSDLKY,103,113,13,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
2,ASGKLVTQW,303,311,12,1.450,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
3,ASWSGKELK,6,14,11,3.300,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*3..."
4,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
...,...,...,...,...,...,...
59,VEDYGFGMF,155,163,10,3.500,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
60,VLWEGGHDL,78,86,13,3.000,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
61,WQIEKASLI,210,218,12,2.000,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
62,YAGPFSQHNY,247,256,11,0.580,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."


In [6]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["qte_de_alelos", "median_binding_percentile"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,FVVDNVHTW,20,28,24,0.685,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
1,CLWPKTHTL,223,231,23,2.100,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
2,SQMLIPKSY,239,247,21,1.400,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*0..."
3,HTWTEQYKF,26,34,21,1.500,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
4,CTMPPLRFL,316,324,20,2.150,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
...,...,...,...,...,...,...
59,RLASAILNA,41,49,10,1.800,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
60,QTVGPWHLGK,263,272,10,2.050,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
61,TPPVSDLKY,105,113,10,2.400,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*0..."
62,RALTPPVSDLK,102,112,10,2.900,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*3..."
